# W-O-SC-01 — NNBT Purpald High-Throughput Screening Assay

### SOP Header (controlled document)

| Field | Value |
|---|---|
| **Doc No** | SOP W-O-SC-01 |
| **Version** | 0.1 (DRAFT) |
| **Effective date** | _TBC_ |
| **Owner** | _TBC (iGEM 2026, DTU)_ |
| **Approved by** | _TBC_ |
| **Workflow ID** | W-O-SC-01 |
| **Workflow name** | NNBT Purpald colorimetric HTS — reagent addition + heater-shaker development |
| **Unit operations** | U-O-01 → U-O-02 → U-O-03 → U-O-04 → U-C-01 |
| **Output** | One assay-ready 96-well plate developed for A₅₃₀ readout, plus the generated, simulator-validated OT-2 `.py` protocol. |

> **Scope.** Supernatant/samples (≈100 µL each) are **pre-loaded** by the operator. This workflow performs only the reagent additions and the heater-shaker development steps, with manual seal/unseal pauses. Absorbance is read off-deck on the Clariostar (U-C-01).


## Steps (unit operations) — read in ~10 s

- **U-O-01** Add NNBT reaction mix (50 µL) to each sample column
- **U-O-02** Seal → shake-incubate (NNBT step), then unseal
- **U-O-03** Add Purpald reagent (50 µL) to each sample column
- **U-O-04** Seal → shake to develop colour, then unseal
- **U-C-01** Read absorbance at 530 nm on the Clariostar (off-deck handoff)


## Run Record

Operator fills this in at run time. Stored with the executed notebook for traceability.


In [1]:
run_record = {
    "run_id": "RUN-SC01-0000",        # PLACEHOLDER
    "operator": "",                    # PLACEHOLDER
    "datetime": "",                    # ISO 8601, filled at run time
    "notes_pre_run": "",
    "notes_during_run": "",
    "notes_deviations": "",
    "notes_post_run": "",
}
run_record


{'run_id': 'RUN-SC01-0000',
 'operator': '',
 'datetime': '',
 'notes_pre_run': '',
 'notes_during_run': '',
 'notes_deviations': '',
 'notes_post_run': ''}

## Parameters — single source of truth

All tunable values live here as named variables **with units in the name**. No magic
numbers anywhere else. Placeholders (e.g. `NUM_SAMPLE_COLUMNS = 10`) are marked.
The generated `.py` protocol bakes these exact values in.


In [2]:
def render_params_block(params):
    """Render the ordered PARAMS list into a PARAMETERS block for the generated .py.
    PARAMS is the single source of truth; the generated file derives from it."""
    lines = ["# " + "=" * 74,
             "# PARAMETERS  (single source of truth - generated from the notebook)",
             "# " + "=" * 74]
    for name, value, comment in params:
        lines.append(f"{name} = {value!r}" + (f"  # {comment}" if comment else ""))
    return "\n".join(lines)

# --- API / hardware ---
API_LEVEL          = "2.16"                # OT-2 API level (Heater-Shaker needs >= 2.13)
P300_MODEL         = "p300_multi_gen2"
P20_MODEL          = "p20_multi_gen2"
P300_MOUNT         = "left"
P20_MOUNT          = "right"

# --- Labware load names ---
ASSAY_PLATE_LOADNAME = "corning_96_wellplate_360ul_flat"   # clear flat-bottom for A530
RESERVOIR_LOADNAME   = "nest_12_reservoir_15ml"
TIPRACK_300_LOADNAME = "opentrons_96_tiprack_300ul"
TIPRACK_20_LOADNAME  = "opentrons_96_tiprack_20ul"

# --- Deck slots (HS adjacency rules: slot 2 stays EMPTY; only tip racks may sit
#     directly front/back of the HS) ---
HS_SLOT          = 1     # Heater-Shaker + assay plate
TIPRACK_300_SLOT = 4     # directly behind HS -> tip rack allowed
RESERVOIR_SLOT   = 5     # diagonal -> allowed
TIPRACK_20_SLOT  = 6
# slot 2 intentionally left empty (left/right-adjacent to HS)

# --- Plate / layout ---
NUM_SAMPLE_COLUMNS  = 10    # PLACEHOLDER: columns of pre-loaded samples (1-12)
PLATE_WELL_MAX_UL   = 360   # corning_96_wellplate_360ul_flat working max

# --- Reagent reservoir wells ---
NNBT_MIX_RESERVOIR_WELL = "A1"   # NNBT reaction mix trough
PURPALD_RESERVOIR_WELL  = "A2"   # Purpald reagent (50 mM in 2N NaOH) trough

# --- Volumes (per well) ---
SUPERNATANT_VOL_UL = 100   # pre-loaded by operator (not pipetted; for volume check)
NNBT_MIX_VOL_UL    = 50
PURPALD_VOL_UL     = 50
TOTAL_VOL_PER_WELL_UL = SUPERNATANT_VOL_UL + NNBT_MIX_VOL_UL + PURPALD_VOL_UL  # 200

# --- Heater-Shaker development ---
# NOTE: OT-2 HS heating floor is 37 C; the Dolz 30 C target CANNOT be actively held.
# Set NNBT_INCUBATION_TEMP_C >= 37 to enable heating, else it shakes at ambient.
NNBT_INCUBATION_TEMP_C   = 30     # target per Dolz 2022 (ambient on OT-2 - see Notes)
NNBT_SHAKE_RPM           = 800
NNBT_INCUBATION_MIN      = 5
DEVELOP_SHAKE_RPM        = 800
DEVELOP_MIN              = 10

# --- Tip strategy ---
NEW_TIP_PER_COLUMN = False   # same reagent into all columns; tips reused (set True to be strict)

# --- Output ---
OUTPUT_DIR     = "."         # generated .py is written alongside this notebook
OUTPUT_PY_NAME = "W-O-SC-01_nnbt_purpald_hts_ot2.py"
WORKFLOW_ID    = "W-O-SC-01"
PROTOCOL_NAME  = "W-O-SC-01 NNBT Purpald HTS"
OWNER          = "iGEM 2026 DTU"
PROTOCOL_DESCRIPTION = ("NNBT colorimetric HTS reagent additions + heater-shaker "
                        "development; samples pre-loaded; read A530 off-deck.")

# Ordered PARAMS list -> baked into the generated .py PARAMETERS block.
PARAMS = [
    ("P300_MODEL", P300_MODEL, ""),
    ("P20_MODEL", P20_MODEL, ""),
    ("P300_MOUNT", P300_MOUNT, ""),
    ("P20_MOUNT", P20_MOUNT, ""),
    ("ASSAY_PLATE_LOADNAME", ASSAY_PLATE_LOADNAME, "clear flat-bottom for A530"),
    ("RESERVOIR_LOADNAME", RESERVOIR_LOADNAME, ""),
    ("TIPRACK_300_LOADNAME", TIPRACK_300_LOADNAME, ""),
    ("TIPRACK_20_LOADNAME", TIPRACK_20_LOADNAME, ""),
    ("HS_SLOT", HS_SLOT, "Heater-Shaker + assay plate"),
    ("TIPRACK_300_SLOT", TIPRACK_300_SLOT, "behind HS (tip rack allowed)"),
    ("RESERVOIR_SLOT", RESERVOIR_SLOT, "diagonal to HS"),
    ("TIPRACK_20_SLOT", TIPRACK_20_SLOT, ""),
    ("NUM_SAMPLE_COLUMNS", NUM_SAMPLE_COLUMNS, "PLACEHOLDER (1-12)"),
    ("NNBT_MIX_RESERVOIR_WELL", NNBT_MIX_RESERVOIR_WELL, ""),
    ("PURPALD_RESERVOIR_WELL", PURPALD_RESERVOIR_WELL, ""),
    ("NNBT_MIX_VOL_UL", NNBT_MIX_VOL_UL, ""),
    ("PURPALD_VOL_UL", PURPALD_VOL_UL, ""),
    ("NNBT_INCUBATION_TEMP_C", NNBT_INCUBATION_TEMP_C, "ambient on OT-2 if <37"),
    ("NNBT_SHAKE_RPM", NNBT_SHAKE_RPM, ""),
    ("NNBT_INCUBATION_MIN", NNBT_INCUBATION_MIN, ""),
    ("DEVELOP_SHAKE_RPM", DEVELOP_SHAKE_RPM, ""),
    ("DEVELOP_MIN", DEVELOP_MIN, ""),
    ("NEW_TIP_PER_COLUMN", NEW_TIP_PER_COLUMN, ""),
]

DECK_SUMMARY = [
    f"slot {HS_SLOT}: Heater-Shaker + assay plate ({ASSAY_PLATE_LOADNAME})",
    "slot 2: EMPTY (HS left/right adjacency rule)",
    f"slot {TIPRACK_300_SLOT}: {TIPRACK_300_LOADNAME}",
    f"slot {RESERVOIR_SLOT}: {RESERVOIR_LOADNAME} (A1=NNBT mix, A2=Purpald)",
    f"slot {TIPRACK_20_SLOT}: {TIPRACK_20_LOADNAME}",
    "slot 12: fixed trash",
]
print("Parameters loaded. Total volume per well:", TOTAL_VOL_PER_WELL_UL, "uL")


Parameters loaded. Total volume per well: 200 uL


## Input validation (pre-flight)

Fail fast before generating anything.


In [3]:
assert 1 <= NUM_SAMPLE_COLUMNS <= 12, "NUM_SAMPLE_COLUMNS must be 1-12"
assert NNBT_MIX_VOL_UL > 0 and PURPALD_VOL_UL > 0, "reagent volumes must be > 0"
assert TOTAL_VOL_PER_WELL_UL <= PLATE_WELL_MAX_UL, "well overfilled"
assert NNBT_MIX_RESERVOIR_WELL != PURPALD_RESERVOIR_WELL, "reagents need distinct troughs"
# Volumes routed to the correct pipette (P20 <=20 uL, P300 20-300 uL):
for v in (NNBT_MIX_VOL_UL, PURPALD_VOL_UL):
    assert 1 <= v <= 300, "volume out of combined pipette range"
print("Input validation: PASS")


Input validation: PASS


## U-O-01 — Add NNBT reaction mix

- **Goal:** Dispense NNBT reaction mix into every pre-loaded sample column.
- **Inputs:** Assay plate (supernatant pre-loaded) on the HS; reservoir `A1` = NNBT reaction mix; P300 multi.
- **Outputs:** Each sample column now holds supernatant + `NNBT_MIX_VOL_UL`.
- **Acceptance criteria:** Latch closed; tips present; `NNBT_MIX_VOL_UL` ≤ 300.
- **Record:** Tip count used; reservoir lot/volume.
- **Deviation handling:** Insufficient mix in trough → pause, top up, resume.
- **Notes:** Same reagent to all columns → one tip set reused (set `NEW_TIP_PER_COLUMN=True` to change tips per column).


In [4]:
def u_o_01_lines():
    return """
    # ---- U-O-01: Add NNBT reaction mix ----
    p300.transfer(
        NNBT_MIX_VOL_UL,
        reservoir[NNBT_MIX_RESERVOIR_WELL],
        sample_cols,
        new_tip=("always" if NEW_TIP_PER_COLUMN else "once"),
    )
"""
print("U-O-01 emitter ready")

U-O-01 emitter ready


## U-O-02 — Seal → shake-incubate (NNBT step) → unseal

- **Goal:** Mix and incubate the NNBT reaction by shaking.
- **Inputs:** Sealed plate on HS.
- **Outputs:** Reacted plate ready for Purpald.
- **Acceptance criteria:** Plate sealed before shaking; shaker reaches `NNBT_SHAKE_RPM`.
- **Record:** Actual RPM, duration, plate temperature.
- **Deviation handling:** If a 30 °C incubation is required, move the sealed plate to a 30 °C incubator for `NNBT_INCUBATION_MIN` (the OT-2 HS cannot hold 30 °C).
- **Notes:** **OT-2 HS heats only ≥ 37 °C.** With the 30 °C target it shakes at ambient; heating is enabled only if `NNBT_INCUBATION_TEMP_C ≥ 37`. Manual seal/unseal via `ctx.pause`.


In [5]:
def u_o_02_lines():
    return """
    # ---- U-O-02: Seal -> shake-incubate (NNBT) -> unseal ----
    ctx.pause("U-O-02: SEAL the plate, then resume to shake-incubate.")
    if NNBT_INCUBATION_TEMP_C >= 37:
        hs_mod.set_and_wait_for_temperature(NNBT_INCUBATION_TEMP_C)
    else:
        ctx.comment("NNBT_INCUBATION_TEMP_C < 37 C: OT-2 HS cannot heat; shaking at ambient.")
    hs_mod.set_and_wait_for_shake_speed(NNBT_SHAKE_RPM)
    ctx.delay(minutes=NNBT_INCUBATION_MIN)
    hs_mod.deactivate_shaker()
    if NNBT_INCUBATION_TEMP_C >= 37:
        hs_mod.deactivate_heater()
    ctx.pause("U-O-02 done: UNSEAL the plate, then resume.")
"""
print("U-O-02 emitter ready")

U-O-02 emitter ready


## U-O-03 — Add Purpald reagent

- **Goal:** Dispense Purpald reagent into every sample column to start colour development.
- **Inputs:** Unsealed plate; reservoir `A2` = Purpald (50 mM in 2N NaOH); P300 multi.
- **Outputs:** Each well at `TOTAL_VOL_PER_WELL_UL`.
- **Acceptance criteria:** Plate unsealed; `PURPALD_VOL_UL` ≤ 300.
- **Record:** Purpald prep time (NaOH freshness), tip count.
- **Deviation handling:** Visible precipitate in trough → discard, re-prepare Purpald, resume.
- **Notes:** **Caustic (2N NaOH)** — gloves/eye protection; segregate tips to caustic waste. Tips reused across columns (same reagent).


In [6]:
def u_o_03_lines():
    return """
    # ---- U-O-03: Add Purpald reagent ----
    p300.transfer(
        PURPALD_VOL_UL,
        reservoir[PURPALD_RESERVOIR_WELL],
        sample_cols,
        new_tip=("always" if NEW_TIP_PER_COLUMN else "once"),
    )
"""
print("U-O-03 emitter ready")

U-O-03 emitter ready


## U-O-04 — Seal → shake to develop → unseal

- **Goal:** Develop the purple Purpald adduct by shaking.
- **Inputs:** Sealed plate on HS.
- **Outputs:** Developed plate ready for A₅₃₀.
- **Acceptance criteria:** Plate sealed; shaker reaches `DEVELOP_SHAKE_RPM`; `DEVELOP_MIN` elapsed.
- **Record:** Actual RPM/time; note time-to-read (colour drifts).
- **Deviation handling:** Over/under-development → record elapsed time; read promptly.
- **Notes:** Room temperature. Latch opened at the end to release the plate.


In [7]:
def u_o_04_lines():
    return """
    # ---- U-O-04: Seal -> shake to develop -> unseal ----
    ctx.pause("U-O-04: SEAL the plate, then resume to develop colour.")
    hs_mod.set_and_wait_for_shake_speed(DEVELOP_SHAKE_RPM)
    ctx.delay(minutes=DEVELOP_MIN)
    hs_mod.deactivate_shaker()
    ctx.pause("U-O-04 done: UNSEAL the plate, then resume.")
"""
print("U-O-04 emitter ready")

U-O-04 emitter ready


## U-C-01 — Read absorbance at 530 nm (Clariostar)

- **Goal:** Quantify colour development at 530 nm.
- **Inputs:** Developed plate.
- **Outputs:** A₅₃₀ values → activity ranking.
- **Acceptance criteria:** Read within the colour-stability window.
- **Record:** Clariostar method file, read time vs. develop end.
- **Deviation handling:** Reader unavailable → keep plate dark, read ASAP, note delay.
- **Notes:** Off-deck on OT-2 (no Clariostar driver). The protocol releases the plate latch and pauses for the operator to transfer it.


In [8]:
def u_c_01_lines():
    return """
    # ---- U-C-01: Hand off to Clariostar (530 nm) ----
    hs_mod.open_labware_latch()
    ctx.pause("U-C-01: Remove the plate and read A530 on the Clariostar.")
"""
print("U-C-01 emitter ready")

# Unit operations executed in order inside run():
UNIT_OP_EMITTERS = [u_o_01_lines, u_o_02_lines, u_o_03_lines, u_o_04_lines, u_c_01_lines]

U-C-01 emitter ready


## Assemble & generate the OT-2 protocol

In [9]:
def setup_lines():
    """Deck + labware + instruments (with Heater-Shaker)."""
    return f"""
    # ---- Deck, labware, instruments ----
    hs_mod = ctx.load_module("heaterShakerModuleV1", HS_SLOT)
    assay_plate = hs_mod.load_labware(ASSAY_PLATE_LOADNAME)
    tiprack_300 = ctx.load_labware(TIPRACK_300_LOADNAME, TIPRACK_300_SLOT)
    reservoir   = ctx.load_labware(RESERVOIR_LOADNAME, RESERVOIR_SLOT)
    tiprack_20  = ctx.load_labware(TIPRACK_20_LOADNAME, TIPRACK_20_SLOT)
    p300 = ctx.load_instrument(P300_MODEL, P300_MOUNT, tip_racks=[tiprack_300])
    p20  = ctx.load_instrument(P20_MODEL,  P20_MOUNT,  tip_racks=[tiprack_20])

    sample_cols = [assay_plate[f"A{{c}}"] for c in range(1, NUM_SAMPLE_COLUMNS + 1)]
    hs_mod.close_labware_latch()  # secure plate for pipetting + shaking
"""

# Assemble the standalone OT-2 protocol from the unit-operation emitters.
import os, textwrap

_metadata = {
    "protocolName": PROTOCOL_NAME,
    "author": OWNER,
    "source": "Generated by " + WORKFLOW_ID + " literate notebook",
    "description": PROTOCOL_DESCRIPTION,
}

_header = (
    "# " + "=" * 74 + "\n"
    "# " + WORKFLOW_ID + "  |  " + PROTOCOL_NAME + "\n"
    "# Auto-generated from the literate notebook - edit PARAMETERS in the\n"
    "# notebook and re-generate; do not hand-edit this file.\n"
    "# " + "=" * 74 + "\n\n"
    "from opentrons import protocol_api\n\n"
    "metadata = " + repr(_metadata) + "\n"
    'requirements = {"robotType": "OT-2", "apiLevel": ' + repr(API_LEVEL) + "}\n\n"
)

_params_block = render_params_block(PARAMS) + "\n\n"

_run_open = "def run(ctx: protocol_api.ProtocolContext):\n"

# Order of unit operations inside run():
_body = setup_lines()
for _emit in UNIT_OP_EMITTERS:
    _body += _emit()

PROTOCOL_SRC = _header + _params_block + _run_open + _body

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PY_PATH = os.path.join(OUTPUT_DIR, OUTPUT_PY_NAME)
with open(OUTPUT_PY_PATH, "w") as _f:
    _f.write(PROTOCOL_SRC)
print("Generated:", OUTPUT_PY_PATH, f"({len(PROTOCOL_SRC)} chars)")


Generated: ./W-O-SC-01_nnbt_purpald_hts_ot2.py (3991 chars)


## Validation gates (output checks + py_compile + opentrons_simulate)

In [10]:
# ---- Validation gates (mandatory before release) ----
import py_compile, subprocess, sys

results = {}

# 1) Output validation: destination wells unique, counts correct, mapping sane.
dest_wells = [f"{row}{col}" for col in range(1, NUM_SAMPLE_COLUMNS + 1)
              for row in "ABCDEFGH"]
results["unique_wells"] = (len(dest_wells) == len(set(dest_wells)))
results["well_count"] = (len(dest_wells) == NUM_SAMPLE_COLUMNS * 8)
results["columns_in_range"] = (1 <= NUM_SAMPLE_COLUMNS <= 12)
results["volume_fits_well"] = (TOTAL_VOL_PER_WELL_UL <= PLATE_WELL_MAX_UL)

# 2) Script validation: py_compile the generated protocol.
try:
    py_compile.compile(OUTPUT_PY_PATH, doraise=True)
    results["py_compile"] = True
except py_compile.PyCompileError as e:
    results["py_compile"] = False
    print(e)

# 3) Simulation validation: opentrons_simulate the generated protocol.
proc = subprocess.run(["opentrons_simulate", OUTPUT_PY_PATH],
                      capture_output=True, text=True)
results["opentrons_simulate"] = (proc.returncode == 0)
if proc.returncode != 0:
    print(proc.stdout[-2000:])
    print(proc.stderr[-2000:])

print("\nValidation results:")
for k, v in results.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

all_pass = all(results.values())
print("\nOVERALL:", "PASS - release OK" if all_pass else "FAIL - do not release")
assert all_pass, "Validation gate failed - see above."



Validation results:
  PASS  unique_wells
  PASS  well_count
  PASS  columns_in_range
  PASS  volume_fits_well
  PASS  py_compile
  PASS  opentrons_simulate

OVERALL: PASS - release OK


## Artifacts & run summary

In [11]:
# ---- Artifacts & run summary ----
print("ARTIFACT (generated OT-2 protocol):")
print("  ", OUTPUT_PY_PATH)
print("\nKey parameters used:")
for name, value, _ in PARAMS:
    print(f"  {name} = {value!r}")
print("\nDeck summary:")
for line in DECK_SUMMARY:
    print("  ", line)


ARTIFACT (generated OT-2 protocol):
   ./W-O-SC-01_nnbt_purpald_hts_ot2.py

Key parameters used:
  P300_MODEL = 'p300_multi_gen2'
  P20_MODEL = 'p20_multi_gen2'
  P300_MOUNT = 'left'
  P20_MOUNT = 'right'
  ASSAY_PLATE_LOADNAME = 'corning_96_wellplate_360ul_flat'
  RESERVOIR_LOADNAME = 'nest_12_reservoir_15ml'
  TIPRACK_300_LOADNAME = 'opentrons_96_tiprack_300ul'
  TIPRACK_20_LOADNAME = 'opentrons_96_tiprack_20ul'
  HS_SLOT = 1
  TIPRACK_300_SLOT = 4
  RESERVOIR_SLOT = 5
  TIPRACK_20_SLOT = 6
  NUM_SAMPLE_COLUMNS = 10
  NNBT_MIX_RESERVOIR_WELL = 'A1'
  PURPALD_RESERVOIR_WELL = 'A2'
  NNBT_MIX_VOL_UL = 50
  PURPALD_VOL_UL = 50
  NNBT_INCUBATION_TEMP_C = 30
  NNBT_SHAKE_RPM = 800
  NNBT_INCUBATION_MIN = 5
  DEVELOP_SHAKE_RPM = 800
  DEVELOP_MIN = 10
  NEW_TIP_PER_COLUMN = False

Deck summary:
   slot 1: Heater-Shaker + assay plate (corning_96_wellplate_360ul_flat)
   slot 2: EMPTY (HS left/right adjacency rule)
   slot 4: opentrons_96_tiprack_300ul
   slot 5: nest_12_reservoir_15ml (A1=N

## Change Log

| Date | Version | Author | Summary of changes |
|---|---|---|---|
| _TBC_ | 0.1 | Claude (draft) | Initial literate notebook generating + simulating the OT-2 NNBT Purpald HTS protocol. |
